# Step 22.3 — GELU → SwiGLU：把内容分支与门控分支分开

## 先建立独立工作副本（以下只是命令参考，本次未执行）

在仓库根目录先同步母版：

```bash
git pull
```

再创建不覆盖已有文件的时间戳副本（仓库应已有 notebooks/work 目录）：

```bash
cp -n notebooks/99_step22_3_swiglu_gated_mlp.ipynb notebooks/work/99_step22_3_swiglu_gated_mlp_$(date +%Y%m%d_%H%M%S)_work.ipynb
```

只在工作副本中填写练习或未来运行。`cp -n` 在同名目标存在时不覆盖；同一秒再次操作也不会覆盖已有内容。当前母版是**材料初稿 0/2**，不是学习完成证明。

**执行边界：**所有代码单元 execution_count 为 null、outputs 为空。本次不执行 notebook、MLX 实验、测试、训练、验证、采样、语料读取/下载或 checkpoint 操作。下面的短实验是将来可观察的安排，不是已观察结果。MLX API 基线固定 v0.32.2。

## 1. 为什么需要这一节

原始 MLP 常写成 Linear(C,4C) → GELU → Linear(4C,C)。Gated MLP 使用两个投影分支，一个生成内容，一个生成门控信号，然后逐元素相乘。SwiGLU 是用 SiLU/Swish 激活门控分支的 GLU 变体，许多现代语言模型采用它。

“门”不是硬开关，也不是注意力权重。本节不声称换一行激活就能无代价变好；结构和参数预算都变了，需要独立训练和比较。

## 2. 先备数学与 Python / MLX 语义

σ(x)=1/(1+exp(−x))，SiLU(x)=xσ(x)。SwiGLU(h)=[SiLU(hWg) ⊙ (hWu)]Wd，其中 ⊙ 为同形状逐元素乘法，Wg,Wu∈R^(C×F)，Wd∈R^(F×C)。MLX Linear 存储 weight 为 (out,in)，调用已处理转置。

无 bias 的标准 MLP 若隐藏宽度为 4C，参数为 8C²；无 bias gated MLP 有三份矩阵，参数为 3CF。要接近原预算，取 F≈8C/3，而不是仍用 4C。实际模型可能按硬件友好的倍数取整，必须再报告真实参数数目。

## 3. 纸笔走一遍（讲解例子，不是练习答案）

先只看 gate=[0,2]、up=[3,−1]。SiLU(0)=0，SiLU(2)=2/(1+e⁻²)，乘积为 [0,−2/(1+e⁻²)]。门控结果可以为负，也不限制在 [0,1]，不能把 SiLU gate 当成概率。

C=6 时，标准 F=24 的两矩阵共有 288 权重；gated F=16 的三矩阵也有 288。这是符号计算，不是实际模型树计数或内存测量。

## 4. Shape tracing 与接口契约

h (B,T,C) → gate/up 各 (B,T,F) → SiLU(gate) (B,T,F) → elementwise product (B,T,F) → down (B,T,C)。MLP 在每个 token 的 feature 轴处理，不混合时间位置；跨 token 交互仍来自 attention。

In [ ]:
import math
import mlx.core as mx

# 本单元及后续所有单元均为未执行的教学源码。
# 仅在未来独立工作副本中按顺序运行；本次材料制作没有导入 MLX。
import mlx.nn as nn

class TinySwiGLU(nn.Module):
    def __init__(self, dims, hidden_dims):
        super().__init__()
        self.gate = nn.Linear(dims, hidden_dims, bias=False)
        self.up = nn.Linear(dims, hidden_dims, bias=False)
        self.down = nn.Linear(hidden_dims, dims, bias=False)

    def __call__(self, x):
        return self.down(nn.silu(self.gate(x)) * self.up(x))

## 5. 短实验计划

先预测 shape 与趋势，再在未来工作副本中运行下一个单元。这里没有运行记录。若未来结果和推导不同，先检查输入、axis、dtype、mask、随机 key，再讨论数值误差；不要把一次随机输出当成分布证明。

In [ ]:
gate = mx.array([[0., 2.]], dtype=mx.float32)
up = mx.array([[3., -1.]], dtype=mx.float32)
product = nn.silu(gate) * up
block = TinySwiGLU(6, 16)
x = mx.zeros((1, 2, 6), dtype=mx.float32)
y = block(x)
mx.eval(product, y)
print("gate product:", product.tolist())
print("MLP shape:", x.shape, "->", y.shape)

## 6. 常见误区与未来衔接

这个随机初始化模块只用于形状观察，不是迁移后的 MiniGPT，也不能说明质量提升。即使两个方案参数数量相同，激活驻留、乘法、kernel fusion、初始化分布都可能不同；不得据参数量宣称吞吐等价。

GLU 常用 sigmoid gate，SwiGLU 用 SiLU gate；GELU 是另一种激活，两者不能只凭名字替换。改变隐藏维度或增加分支后，旧权重和 Adam moments 的 shape/path 都不匹配，应保留原 baseline，另建实验配置。

## 7. 练习（不附答案）

1. C=12 时求理论同预算 F，并分别写出三个 MLX weight shape。
2. 解释逐元素乘法与矩阵乘法在这里各自出现的位置。
3. 考虑保留 bias 的方案，重写两个 MLP 的参数公式，不把无 bias 公式直接套用。

请先用纸笔作答，再决定是否在工作副本中添加自己的代码。母版不填入参考答案。

## 8. 官方 API 阅读与完成边界

- [MLX v0.32.2 / python/mlx/nn/layers/activations.py](https://github.com/ml-explore/mlx/blob/v0.32.2/python/mlx/nn/layers/activations.py)

接口说明以固定版本源码为准。本次只有文本与源代码静态检查，没有运行通过声明。能够复述概念与完成练习后再进入下一节；材料覆盖范围不代表读者已经掌握。